In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from math import radians, sin, cos, sqrt, atan2

In [3]:
routestops = pd.read_csv("route_stops_full.csv")
routestops.head()

,PublishedLineName,DirectionRef,Stop Name,stoplat,stoplon,obs
0,B35,0.0,14 AV/36 ST,40.641006,-73.982306,925
1,B35,0.0,14 AV/39 ST,40.639481,-73.984300,1408
2,B35,0.0,14 AV/CHURCH AV,40.641736,-73.981549,1037
3,B35,0.0,39 ST/10 AV,40.644695,-73.993030,784
4,B35,0.0,39 ST/12 AV,40.642054,-73.988660,997


In [4]:
print(routestops.columns.tolist())

['PublishedLineName', 'DirectionRef', 'Stop Name', 'stoplat', 'stoplon', 'obs']


In [5]:
required_cols = ["PublishedLineName", "DirectionRef", "Stop Name", "stoplat", "stoplon", "obs"]
missing_cols = [c for c in required_cols if c not in routestops.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

routestops = routestops[required_cols].copy()
routestops = routestops.dropna(subset=["PublishedLineName", "DirectionRef", "Stop Name", "stoplat", "stoplon"])
routestops["DirectionRef"] = routestops["DirectionRef"].astype(float)
routestops["Stop Name"] = routestops["Stop Name"].astype(str).str.strip()

In [6]:
def haversine(lat1, lon1, lat2, lon2, R=6371000):
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = (
        sin(dlat / 2) ** 2
        + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    )
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

In [7]:
def order_route_direction(group):
    group = group.drop_duplicates(subset=["Stop Name"]).copy()

    lonspan = group["stoplon"].max() - group["stoplon"].min()
    latspan = group["stoplat"].max() - group["stoplat"].min()

    if lonspan >= latspan:
        group = group.sort_values("stoplon", ascending=(group["DirectionRef"].iloc[0] == 0.0))
    else:
        group = group.sort_values("stoplat", ascending=(group["DirectionRef"].iloc[0] == 0.0))

    return group.reset_index(drop=True)

In [8]:
ordered_groups = []

for (route, direction), group in routestops.groupby(["PublishedLineName", "DirectionRef"], sort=True):
    ordered = order_route_direction(group)
    ordered_groups.append(ordered)

routestops_ordered = pd.concat(ordered_groups, ignore_index=True)
routestops_ordered.head(20)

,PublishedLineName,DirectionRef,Stop Name,stoplat,stoplon,obs
0,B35,0.0,39 ST/2 AV,40.655490,-74.010854,1971
1,B35,0.0,39 ST/3 AV,40.654138,-74.008598,1854
2,B35,0.0,39 ST/4 AV,40.652815,-74.006473,1527
3,B35,0.0,39 ST/5 AV,40.651442,-74.004200,2645
4,B35,0.0,39 ST/6 AV,40.650020,-74.001847,553
5,B35,0.0,39 ST/7 AV,40.648699,-73.999661,821
6,B35,0.0,39 ST/8 AV,40.647353,-73.997436,979
7,B35,0.0,39 ST/9 AV,40.646011,-73.995206,1627
8,B35,0.0,39 ST/10 AV,40.644695,-73.993030,784
9,B35,0.0,39 ST/FT HAMILTON PY,40.643319,-73.990750,1906


In [9]:
state_nodes = routestops_ordered.copy()
state_nodes = state_nodes.rename(columns={
    "PublishedLineName": "route",
    "DirectionRef": "direction",
    "Stop Name": "stop_name"
})

state_nodes["state_id"] = (
    state_nodes["route"].astype(str) + "_" +
    state_nodes["direction"].astype(str) + "_" +
    state_nodes["stop_name"].astype(str)
)

state_nodes = state_nodes[["state_id", "route", "direction", "stop_name", "stoplat", "stoplon", "obs"]].copy()
state_nodes.head()

,state_id,route,direction,stop_name,stoplat,stoplon,obs
0,B35_0.0_39 ST/2 AV,B35,0.0,39 ST/2 AV,40.655490,-74.010854,1971
1,B35_0.0_39 ST/3 AV,B35,0.0,39 ST/3 AV,40.654138,-74.008598,1854
2,B35_0.0_39 ST/4 AV,B35,0.0,39 ST/4 AV,40.652815,-74.006473,1527
3,B35_0.0_39 ST/5 AV,B35,0.0,39 ST/5 AV,40.651442,-74.004200,2645
4,B35_0.0_39 ST/6 AV,B35,0.0,39 ST/6 AV,40.650020,-74.001847,553


In [10]:
ride_edges = []

for (route, direction), group in state_nodes.groupby(["route", "direction"], sort=True):
    group = group.reset_index(drop=True)

    for i in range(len(group) - 1):
        a = group.iloc[i]
        b = group.iloc[i + 1]

        dist_m = haversine(a["stoplat"], a["stoplon"], b["stoplat"], b["stoplon"])

        if dist_m > 3000:
            continue

        sched_time_min = dist_m / 250.0

        ride_edges.append({
            "route": route,
            "direction": direction,
            "from_stop": a["stop_name"],
            "to_stop": b["stop_name"],
            "from_state": a["state_id"],
            "to_state": b["state_id"],
            "distance_m": dist_m,
            "from_obs": a["obs"],
            "to_obs": b["obs"],
            "sched_time_min": sched_time_min,
            "mean_delay_min": 0.0,
            "median_delay_min": 0.0,
            "std_delay_min": 0.0,
            "n_delay_obs": 0,
            "risk_score": 0.0,
            "service_strength": float(min(a["obs"], b["obs"])),
            "generalized_cost": sched_time_min,
            "edge_type": "ride"
        })

ride_edges_df = pd.DataFrame(ride_edges)
ride_edges_df.head()

,route,direction,from_stop,to_stop,from_state,to_state,distance_m,from_obs,to_obs,sched_time_min,mean_delay_min,median_delay_min,std_delay_min,n_delay_obs,risk_score,service_strength,generalized_cost,edge_type
0,B35,0.0,39 ST/2 AV,39 ST/3 AV,B35_0.0_39 ST/2 AV,B35_0.0_39 ST/3 AV,242.559736,1971,1854,0.970239,0.0,0.0,0.0,0,0.0,1854.0,0.970239,ride
1,B35,0.0,39 ST/3 AV,39 ST/4 AV,B35_0.0_39 ST/3 AV,B35_0.0_39 ST/4 AV,231.866520,1854,1527,0.927466,0.0,0.0,0.0,0,0.0,1527.0,0.927466,ride
2,B35,0.0,39 ST/4 AV,39 ST/5 AV,B35_0.0_39 ST/4 AV,B35_0.0_39 ST/5 AV,245.107279,1527,2645,0.980429,0.0,0.0,0.0,0,0.0,1527.0,0.980429,ride
3,B35,0.0,39 ST/5 AV,39 ST/6 AV,B35_0.0_39 ST/5 AV,B35_0.0_39 ST/6 AV,253.784113,2645,553,1.015136,0.0,0.0,0.0,0,0.0,553.0,1.015136,ride
4,B35,0.0,39 ST/6 AV,39 ST/7 AV,B35_0.0_39 ST/6 AV,B35_0.0_39 ST/7 AV,235.769930,553,821,0.943080,0.0,0.0,0.0,0,0.0,553.0,0.943080,ride


In [11]:
TRANSFER_PENALTY_MIN = 8.0

transfer_rows = []

for stop_name, group in state_nodes.groupby("stop_name"):
    records = group[["state_id", "route", "direction", "stop_name"]].drop_duplicates().to_dict("records")

    for i in range(len(records)):
        for j in range(len(records)):
            if i == j:
                continue

            a = records[i]
            b = records[j]

            if a["route"] == b["route"]:
                continue

            transfer_rows.append({
                "route": "TRANSFER",
                "direction": -1,
                "from_stop": stop_name,
                "to_stop": stop_name,
                "from_state": a["state_id"],
                "to_state": b["state_id"],
                "distance_m": 0.0,
                "from_obs": 0.0,
                "to_obs": 0.0,
                "sched_time_min": 0.0,
                "mean_delay_min": 0.0,
                "median_delay_min": 0.0,
                "std_delay_min": 0.0,
                "n_delay_obs": 0,
                "risk_score": 0.0,
                "service_strength": 0.0,
                "generalized_cost": TRANSFER_PENALTY_MIN,
                "edge_type": "transfer"
            })

transfer_edges_df = pd.DataFrame(transfer_rows)
transfer_edges_df.head()

""


In [12]:
physical_stops = (
    state_nodes.groupby("stop_name", as_index=False)
    .agg(
        stoplat=("stoplat", "median"),
        stoplon=("stoplon", "median")
    )
)

physical_stops.head()

,stop_name,stoplat,stoplon
0,108 ST/53 AV,40.742945,-73.854753
1,108 ST/HORACE HARDING EP,40.737824,-73.851798
2,108 ST/HORACE HARDING EXPWY,40.738070,-73.852257
3,108 ST/MARTENSE AV,40.742284,-73.854417
4,108 ST/OTIS AV,40.741549,-73.854038


In [13]:
MAX_WALK_DISTANCE_M = 400
WALK_SPEED_M_PER_S = 1.3
WALK_PENALTY_MIN = 2.0
MAX_WALK_NEIGHBORS = 3

In [14]:
walk_pairs = []

stops_records = physical_stops.to_dict("records")

for i in range(len(stops_records)):
    a = stops_records[i]
    candidates = []

    for j in range(len(stops_records)):
        if i == j:
            continue

        b = stops_records[j]
        dist_m = haversine(a["stoplat"], a["stoplon"], b["stoplat"], b["stoplon"])

        if 0 < dist_m <= MAX_WALK_DISTANCE_M:
            candidates.append((b["stop_name"], dist_m))

    candidates = sorted(candidates, key=lambda x: x[1])[:MAX_WALK_NEIGHBORS]

    for to_stop, dist_m in candidates:
        walk_time_min = dist_m / WALK_SPEED_M_PER_S / 60.0

        walk_pairs.append({
            "from_stop": a["stop_name"],
            "to_stop": to_stop,
            "walk_distance_m": dist_m,
            "walk_time_min": walk_time_min,
            "walk_cost": walk_time_min + WALK_PENALTY_MIN
        })

walk_pairs_df = pd.DataFrame(walk_pairs).drop_duplicates()
walk_pairs_df.head()

,from_stop,to_stop,walk_distance_m,walk_time_min,walk_cost
0,108 ST/53 AV,CORONA AV/51 AV,76.154602,0.976341,2.976341
1,108 ST/53 AV,108 ST/MARTENSE AV,78.762353,1.009774,3.009774
2,108 ST/53 AV,108 ST/OTIS AV,166.505960,2.134692,4.134692
3,108 ST/HORACE HARDING EP,108 ST/HORACE HARDING EXPWY,47.368340,0.607286,2.607286
4,108 ST/HORACE HARDING EP,108 ST/WALDRON ST,178.963167,2.294400,4.294400


In [15]:
walk_edge_rows = []

state_lookup = (
    state_nodes.groupby("stop_name")["state_id"]
    .apply(list)
    .to_dict()
)

route_lookup = (
    state_nodes.groupby("state_id")[["route", "direction"]]
    .first()
)

for _, row in walk_pairs_df.iterrows():
    from_states = state_lookup.get(row["from_stop"], [])
    to_states = state_lookup.get(row["to_stop"], [])

    for fs in from_states:
        for ts in to_states:
            if fs == ts:
                continue

            fs_route = route_lookup.loc[fs, "route"]
            ts_route = route_lookup.loc[ts, "route"]

            if fs_route == ts_route:
                continue

            walk_edge_rows.append({
                "route": "WALK",
                "direction": -2,
                "from_stop": row["from_stop"],
                "to_stop": row["to_stop"],
                "from_state": fs,
                "to_state": ts,
                "distance_m": row["walk_distance_m"],
                "from_obs": 0.0,
                "to_obs": 0.0,
                "sched_time_min": row["walk_time_min"],
                "mean_delay_min": 0.0,
                "median_delay_min": 0.0,
                "std_delay_min": 0.0,
                "n_delay_obs": 0,
                "risk_score": 0.0,
                "service_strength": 0.0,
                "generalized_cost": row["walk_cost"],
                "edge_type": "walk"
            })

walk_edges_df = pd.DataFrame(walk_edge_rows)
walk_edges_df.head()

,route,direction,from_stop,to_stop,from_state,to_state,distance_m,from_obs,to_obs,sched_time_min,mean_delay_min,median_delay_min,std_delay_min,n_delay_obs,risk_score,service_strength,generalized_cost,edge_type
0,WALK,-2,41 RD/MAIN ST,MAIN ST/KISSENA BL,Q58_0.0_41 RD/MAIN ST,Q44-SBS_0.0_MAIN ST/KISSENA BL,82.968777,0.0,0.0,1.063702,0.0,0.0,0.0,0,0.0,0.0,3.063702,walk
1,WALK,-2,41 RD/MAIN ST,MAIN ST/41 AV,Q58_0.0_41 RD/MAIN ST,Q44-SBS_1.0_MAIN ST/41 AV,119.979214,0.0,0.0,1.538195,0.0,0.0,0.0,0,0.0,0.0,3.538195,walk
2,WALK,-2,41 RD/MAIN ST,MAIN ST/39 AV,Q58_0.0_41 RD/MAIN ST,Q44-SBS_0.0_MAIN ST/39 AV,340.844254,0.0,0.0,4.369798,0.0,0.0,0.0,0,0.0,0.0,6.369798,walk
3,WALK,-2,41 RD/MAIN ST,MAIN ST/39 AV,Q58_0.0_41 RD/MAIN ST,Q44-SBS_1.0_MAIN ST/39 AV,340.844254,0.0,0.0,4.369798,0.0,0.0,0.0,0,0.0,0.0,6.369798,walk
4,WALK,-2,CHURCH AV/BEDFORD AV,FLATBUSH AV/CHURCH AV,B35_0.0_CHURCH AV/BEDFORD AV,B41_0.0_FLATBUSH AV/CHURCH AV,234.404108,0.0,0.0,3.005181,0.0,0.0,0.0,0,0.0,0.0,5.005181,walk


In [16]:
all_state_edges = pd.concat(
    [ride_edges_df, transfer_edges_df, walk_edges_df],
    ignore_index=True,
    sort=False
)

all_state_edges.head()

,route,direction,from_stop,to_stop,from_state,to_state,distance_m,from_obs,to_obs,sched_time_min,mean_delay_min,median_delay_min,std_delay_min,n_delay_obs,risk_score,service_strength,generalized_cost,edge_type
0,B35,0.0,39 ST/2 AV,39 ST/3 AV,B35_0.0_39 ST/2 AV,B35_0.0_39 ST/3 AV,242.559736,1971.0,1854.0,0.970239,0.0,0.0,0.0,0,0.0,1854.0,0.970239,ride
1,B35,0.0,39 ST/3 AV,39 ST/4 AV,B35_0.0_39 ST/3 AV,B35_0.0_39 ST/4 AV,231.866520,1854.0,1527.0,0.927466,0.0,0.0,0.0,0,0.0,1527.0,0.927466,ride
2,B35,0.0,39 ST/4 AV,39 ST/5 AV,B35_0.0_39 ST/4 AV,B35_0.0_39 ST/5 AV,245.107279,1527.0,2645.0,0.980429,0.0,0.0,0.0,0,0.0,1527.0,0.980429,ride
3,B35,0.0,39 ST/5 AV,39 ST/6 AV,B35_0.0_39 ST/5 AV,B35_0.0_39 ST/6 AV,253.784113,2645.0,553.0,1.015136,0.0,0.0,0.0,0,0.0,553.0,1.015136,ride
4,B35,0.0,39 ST/6 AV,39 ST/7 AV,B35_0.0_39 ST/6 AV,B35_0.0_39 ST/7 AV,235.769930,553.0,821.0,0.943080,0.0,0.0,0.0,0,0.0,553.0,0.943080,ride


In [17]:
G2 = nx.DiGraph()

for _, row in state_nodes.iterrows():
    G2.add_node(
        row["state_id"],
        stop_name=row["stop_name"],
        route=row["route"],
        direction=row["direction"],
        stoplat=row["stoplat"],
        stoplon=row["stoplon"],
        obs=row["obs"]
    )

for _, row in all_state_edges.iterrows():
    G2.add_edge(
        row["from_state"],
        row["to_state"],
        edge_type=row["edge_type"],
        route=row["route"],
        direction=row["direction"],
        from_stop=row["from_stop"],
        to_stop=row["to_stop"],
        distance_m=row["distance_m"],
        sched_time_min=row["sched_time_min"],
        mean_delay_min=row["mean_delay_min"],
        median_delay_min=row["median_delay_min"],
        std_delay_min=row["std_delay_min"],
        risk_score=row["risk_score"],
        generalized_cost=row["generalized_cost"]
    )

print("State nodes:", G2.number_of_nodes())
print("State edges:", G2.number_of_edges())

State nodes: 540
State edges: 584


In [18]:
def shortest_path_same_route_direction(G, state_nodes_df, origin_stop, destination_stop, route, direction, weight="generalized_cost"):
    origin_candidates = state_nodes_df[
        (state_nodes_df["stop_name"] == origin_stop) &
        (state_nodes_df["route"] == route) &
        (state_nodes_df["direction"] == direction)
    ]["state_id"].tolist()

    destination_candidates = state_nodes_df[
        (state_nodes_df["stop_name"] == destination_stop) &
        (state_nodes_df["route"] == route) &
        (state_nodes_df["direction"] == direction)
    ]["state_id"].tolist()

    best_result = None

    for o in origin_candidates:
        for d in destination_candidates:
            try:
                path = nx.shortest_path(G, o, d, weight=weight)
                cost = nx.shortest_path_length(G, o, d, weight=weight)

                if best_result is None or cost < best_result["cost"]:
                    best_result = {
                        "origin_state": o,
                        "destination_state": d,
                        "path": path,
                        "cost": cost
                    }
            except nx.NetworkXNoPath:
                continue

    return best_result

In [19]:
def best_path_between_stops_prefer_direct(G, state_nodes_df, origin_stop, destination_stop, weight="generalized_cost"):
    origin_rows = state_nodes_df[state_nodes_df["stop_name"] == origin_stop].copy()
    destination_rows = state_nodes_df[state_nodes_df["stop_name"] == destination_stop].copy()

    if origin_rows.empty or destination_rows.empty:
        return None

    shared_pairs = (
        origin_rows[["route", "direction"]]
        .merge(destination_rows[["route", "direction"]], on=["route", "direction"])
        .drop_duplicates()
    )

    direct_best = None

    for _, pair in shared_pairs.iterrows():
        route = pair["route"]
        direction = pair["direction"]

        result = shortest_path_same_route_direction(
            G, state_nodes_df, origin_stop, destination_stop, route, direction, weight=weight
        )

        if result is not None:
            if direct_best is None or result["cost"] < direct_best["cost"]:
                direct_best = result

    if direct_best is not None:
        direct_best["mode"] = "direct_same_route_direction"
        return direct_best

    origins = origin_rows["state_id"].tolist()
    destinations = destination_rows["state_id"].tolist()

    best_result = None

    for o in origins:
        for d in destinations:
            if o == d:
                continue
            try:
                path = nx.shortest_path(G, o, d, weight=weight)
                cost = nx.shortest_path_length(G, o, d, weight=weight)

                if best_result is None or cost < best_result["cost"]:
                    best_result = {
                        "origin_state": o,
                        "destination_state": d,
                        "path": path,
                        "cost": cost,
                        "mode": "multimodal"
                    }
            except nx.NetworkXNoPath:
                continue

    return best_result

In [20]:
def describe_path(G, path):
    rows = []

    for i in range(len(path) - 1):
        u = path[i]
        v = path[i + 1]
        data = G[u][v]

        rows.append({
            "from_state": u,
            "to_state": v,
            "edge_type": data.get("edge_type"),
            "route": data.get("route"),
            "direction": data.get("direction"),
            "from_stop": data.get("from_stop"),
            "to_stop": data.get("to_stop"),
            "distance_m": round(float(data.get("distance_m", 0.0)), 2),
            "sched_time_min": round(float(data.get("sched_time_min", 0.0)), 2),
            "generalized_cost": round(float(data.get("generalized_cost", 0.0)), 2)
        })

    return pd.DataFrame(rows)

In [21]:
best_result = best_path_between_stops_prefer_direct(
    G2,
    state_nodes,
    origin_stop="14 AV/36 ST",
    destination_stop="39 ST/12 AV"
)

best_result

{'origin_state': 'B35_0.0_14 AV/36 ST',
 'destination_state': 'B35_1.0_39 ST/12 AV',
 'path': ['B35_0.0_14 AV/36 ST',
  'B35_0.0_14 AV/CHURCH AV',
  'B35_0.0_CHURCH AV/MC DONALD AV',
  'B35_0.0_CHURCH AV/E 3 ST',
  'B35_0.0_CHURCH AV/E 5 ST',
  'B35_0.0_CHURCH AV/OCEAN PY',
  'B35_0.0_CHURCH AV/CONEY ISLAND AV',
  'B35_0.0_CHURCH AV/WESTMINSTER RD',
  'B35_0.0_CHURCH AV/RUGBY RD',
  'B35_0.0_CHURCH AV/E 18 ST',
  'B35_0.0_CHURCH AV/OCEAN AV',
  'B35_0.0_CHURCH AV/FLATBUSH AV',
  'B41_0.0_FLATBUSH AV/CHURCH AV',
  'B35_1.0_CHURCH AV/FLATBUSH AV',
  'B35_1.0_CHURCH AV/OCEAN AV',
  'B35_1.0_CHURCH AV/E 18 ST',
  'B35_1.0_CHURCH AV/E 16 ST',
  'B35_1.0_CHURCH AV/RUGBY RD',
  'B35_1.0_CHURCH AV/WESTMINSTER RD',
  'B35_1.0_CHURCH AV/CONEY ISLAND AV',
  'B35_1.0_CHURCH AV/E 7 ST',
  'B35_1.0_CHURCH AV/E 5 ST',
  'B35_1.0_CHURCH AV/E 3 ST',
  'B35_1.0_CHURCH AV/MC DONALD AV',
  'B35_1.0_CHURCH AV/STORY ST',
  'B35_1.0_CHURCH AV/CHESTER AV',
  'B35_1.0_13 AV/37 ST',
  'B35_1.0_39 ST/13 AV',
  '

In [22]:
best_result["path"]

['B35_0.0_14 AV/36 ST',
 'B35_0.0_14 AV/CHURCH AV',
 'B35_0.0_CHURCH AV/MC DONALD AV',
 'B35_0.0_CHURCH AV/E 3 ST',
 'B35_0.0_CHURCH AV/E 5 ST',
 'B35_0.0_CHURCH AV/OCEAN PY',
 'B35_0.0_CHURCH AV/CONEY ISLAND AV',
 'B35_0.0_CHURCH AV/WESTMINSTER RD',
 'B35_0.0_CHURCH AV/RUGBY RD',
 'B35_0.0_CHURCH AV/E 18 ST',
 'B35_0.0_CHURCH AV/OCEAN AV',
 'B35_0.0_CHURCH AV/FLATBUSH AV',
 'B41_0.0_FLATBUSH AV/CHURCH AV',
 'B35_1.0_CHURCH AV/FLATBUSH AV',
 'B35_1.0_CHURCH AV/OCEAN AV',
 'B35_1.0_CHURCH AV/E 18 ST',
 'B35_1.0_CHURCH AV/E 16 ST',
 'B35_1.0_CHURCH AV/RUGBY RD',
 'B35_1.0_CHURCH AV/WESTMINSTER RD',
 'B35_1.0_CHURCH AV/CONEY ISLAND AV',
 'B35_1.0_CHURCH AV/E 7 ST',
 'B35_1.0_CHURCH AV/E 5 ST',
 'B35_1.0_CHURCH AV/E 3 ST',
 'B35_1.0_CHURCH AV/MC DONALD AV',
 'B35_1.0_CHURCH AV/STORY ST',
 'B35_1.0_CHURCH AV/CHESTER AV',
 'B35_1.0_13 AV/37 ST',
 'B35_1.0_39 ST/13 AV',
 'B35_1.0_39 ST/12 AV']

In [23]:
describe_path(G2, best_result["path"])

,from_state,to_state,edge_type,route,direction,from_stop,to_stop,distance_m,sched_time_min,generalized_cost
0,B35_0.0_14 AV/36 ST,B35_0.0_14 AV/CHURCH AV,ride,B35,0.0,14 AV/36 ST,14 AV/CHURCH AV,103.29,0.41,0.41
1,B35_0.0_14 AV/CHURCH AV,B35_0.0_CHURCH AV/MC DONALD AV,ride,B35,0.0,14 AV/CHURCH AV,CHURCH AV/MC DONALD AV,163.87,0.66,0.66
2,B35_0.0_CHURCH AV/MC DONALD AV,B35_0.0_CHURCH AV/E 3 ST,ride,B35,0.0,CHURCH AV/MC DONALD AV,CHURCH AV/E 3 ST,208.84,0.84,0.84
3,B35_0.0_CHURCH AV/E 3 ST,B35_0.0_CHURCH AV/E 5 ST,ride,B35,0.0,CHURCH AV/E 3 ST,CHURCH AV/E 5 ST,170.32,0.68,0.68
4,B35_0.0_CHURCH AV/E 5 ST,B35_0.0_CHURCH AV/OCEAN PY,ride,B35,0.0,CHURCH AV/E 5 ST,CHURCH AV/OCEAN PY,109.51,0.44,0.44
5,B35_0.0_CHURCH AV/OCEAN PY,B35_0.0_CHURCH AV/CONEY ISLAND AV,ride,B35,0.0,CHURCH AV/OCEAN PY,CHURCH AV/CONEY ISLAND AV,345.78,1.38,1.38
6,B35_0.0_CHURCH AV/CONEY ISLAND AV,B35_0.0_CHURCH AV/WESTMINSTER RD,ride,B35,0.0,CHURCH AV/CONEY ISLAND AV,CHURCH AV/WESTMINSTER RD,234.83,0.94,0.94
7,B35_0.0_CHURCH AV/WESTMINSTER RD,B35_0.0_CHURCH AV/RUGBY RD,ride,B35,0.0,CHURCH AV/WESTMINSTER RD,CHURCH AV/RUGBY RD,171.13,0.68,0.68
8,B35_0.0_CHURCH AV/RUGBY RD,B35_0.0_CHURCH AV/E 18 ST,ride,B35,0.0,CHURCH AV/RUGBY RD,CHURCH AV/E 18 ST,323.96,1.30,1.30
9,B35_0.0_CHURCH AV/E 18 ST,B35_0.0_CHURCH AV/OCEAN AV,ride,B35,0.0,CHURCH AV/E 18 ST,CHURCH AV/OCEAN AV,209.69,0.84,0.84


In [24]:
test_result = shortest_path_same_route_direction(
    G2,
    state_nodes,
    origin_stop="14 AV/36 ST",
    destination_stop="39 ST/12 AV",
    route="B35",
    direction=0.0
)

test_result

In [26]:
b35_dir0 = state_nodes[
    (state_nodes["route"] == "B35") &
    (state_nodes["direction"] == 0.0)
].copy()

print("B35 dir 0 stops:", len(b35_dir0))
b35_dir0[["stop_name", "stoplat", "stoplon"]].reset_index(drop=True)

B35 dir 0 stops: 53


,stop_name,stoplat,stoplon
0,39 ST/2 AV,40.655490,-74.010854
1,39 ST/3 AV,40.654138,-74.008598
2,39 ST/4 AV,40.652815,-74.006473
3,39 ST/5 AV,40.651442,-74.004200
4,39 ST/6 AV,40.650020,-74.001847
5,39 ST/7 AV,40.648699,-73.999661
6,39 ST/8 AV,40.647353,-73.997436
7,39 ST/9 AV,40.646011,-73.995206
8,39 ST/10 AV,40.644695,-73.993030
9,39 ST/FT HAMILTON PY,40.643319,-73.990750


In [27]:
origin_stop = "14 AV/36 ST"
destination_stop = "39 ST/12 AV"

print("Origin in B35 dir0:", origin_stop in set(b35_dir0["stop_name"]))
print("Destination in B35 dir0:", destination_stop in set(b35_dir0["stop_name"]))

Origin in B35 dir0: True
Destination in B35 dir0: True


In [28]:
b35_dir0_edges = ride_edges_df[
    (ride_edges_df["route"] == "B35") &
    (ride_edges_df["direction"] == 0.0)
].copy()

print("B35 dir 0 ride edges:", len(b35_dir0_edges))
b35_dir0_edges[["from_stop", "to_stop", "distance_m"]].reset_index(drop=True)

B35 dir 0 ride edges: 52


,from_stop,to_stop,distance_m
0,39 ST/2 AV,39 ST/3 AV,242.559736
1,39 ST/3 AV,39 ST/4 AV,231.866520
2,39 ST/4 AV,39 ST/5 AV,245.107279
3,39 ST/5 AV,39 ST/6 AV,253.784113
4,39 ST/6 AV,39 ST/7 AV,235.769930
5,39 ST/7 AV,39 ST/8 AV,240.078445
6,39 ST/8 AV,39 ST/9 AV,240.134735
7,39 ST/9 AV,39 ST/10 AV,234.772276
8,39 ST/10 AV,39 ST/FT HAMILTON PY,245.795388
9,39 ST/FT HAMILTON PY,39 ST/12 AV,225.569189


In [29]:
test_result = shortest_path_same_route_direction(
    G2,
    state_nodes,
    origin_stop="14 AV/36 ST",
    destination_stop="39 ST/12 AV",
    route="B35",
    direction=0.0
)

print(test_result)

None


In [30]:
if test_result is None:
    print("No direct path found on B35 direction 0.0")
else:
    display(describe_path(G2, test_result["path"]))

No direct path found on B35 direction 0.0


In [ ]:
state_nodes.to_csv("layer2_state_nodes.csv", index=False)
all_state_edges.to_csv("layer2_state_edges.csv", index=False)

In [ ]:
nx.write_gexf(G2, "layer2_resilient_bus_graph.gexf")

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "state_nodes",
        "ride_edges",
        "transfer_edges",
        "walk_edges",
        "all_edges"
    ],
    "value": [
        len(state_nodes),
        len(ride_edges_df),
        len(transfer_edges_df),
        len(walk_edges_df),
        len(all_state_edges)
    ]
})

summary